In [1]:
import os
import sys
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
# Set the parent directory as the current directory
os.chdir(parent_dir)

In [4]:
from rdma.utils.data import read_json_file
def apply_corrections_to_initial_dataset(
    initial_dataset: dict,
    human_corrections: dict = None,
    hybrid_corrections: dict = None,
    similarity_threshold: int = 85
) -> dict:
    """
    Apply human corrections to override the initial dataset structure using SET-BASED OR logic.
    
    Process flow:
    1. Start with initial dataset structure
    2. Collect all corrections and group by (document_id, normalized_entity)
    3. Apply OR logic: if ANY correction marks an entity as rare_disease=True, keep it as True
    4. Only remove annotations if ALL corrections for that entity mark it as rare_disease=False
    5. Return modified dataset with same structure as initial
    
    Args:
        initial_dataset: Original dataset in initial format
        human_corrections: Human-only corrections file  
        hybrid_corrections: Supervisor+human hybrid corrections file
        similarity_threshold: Threshold for entity clustering
        
    Returns:
        Modified dataset with corrections applied using OR logic, maintaining initial structure
    """
    import copy
    import re
    from collections import defaultdict
    
    # Dictionary for abbreviation expansion (from your original code)
    abbreviations = {
        "hit": "heparin induced thrombocytopenia",
        "pah": "pulmonary arterial hypertension",
        "pfo": "patent foramen ovale",
        "pcd": "primary ciliary dyskinesia",
        "hids": "hyper-igd syndrome",
        "ald": "adrenoleukodystrophy",
    }
    
    def normalize_entity(entity: str) -> str:
        """Normalize entity text for comparison."""
        if not entity:
            return ""
        normalized = entity.lower().strip()
        normalized = re.sub(r'\s+', ' ', normalized)
        normalized = re.sub(r'(\w)-(\w)', r'\1 \2', normalized)
        if abbreviations and normalized in abbreviations:
            return abbreviations[normalized]
        return normalized
    
    def is_valid_annotation(entity, context=""):
        """Check if an annotation is valid."""
        excluded_terms = ["high altitude pulmonary edema"]
        if any(term.lower() in entity.lower() for term in excluded_terms):
            return False
        return True
    
    # Create a deep copy of the initial dataset to avoid modifying the original
    modified_dataset = copy.deepcopy(initial_dataset)
    
    print("=== APPLYING SET-BASED CORRECTIONS WITH OR LOGIC ===")
    print(f"Initial dataset documents: {len(initial_dataset)}")
    print("Note: Using OR logic - if ANY correction marks entity as rare=True, it stays True")
    
    # Step 1: Collect ALL corrections from both sources and group by (doc_id, normalized_entity)
    entity_corrections = defaultdict(list)  # (doc_id, normalized_entity) -> list of corrections
    
    def collect_corrections(corrections, source_name):
        """Collect corrections and group by entity."""
        if not corrections:
            print(f"No {source_name} corrections to collect")
            return
        
        print(f"\n--- Collecting {source_name} corrections ---")
        
        # Determine the structure of corrections
        annotations_to_process = []
        
        if 'corrected_annotations' in corrections:
            # Human corrections format
            annotations_to_process = corrections['corrected_annotations']
            print(f"Found {len(annotations_to_process)} {source_name} annotations")
        elif 'results' in corrections:
            # Supervisor format - collect from all categories
            for category in ['true_positives', 'false_negatives', 'false_positives']:
                if category in corrections['results']:
                    annotations_to_process.extend(corrections['results'][category])
            print(f"Found {len(annotations_to_process)} {source_name} annotations from supervisor format")
        else:
            print(f"Unknown format for {source_name} corrections")
            return
        
        collected_count = 0
        for annotation in annotations_to_process:
            if not all(key in annotation for key in ['entity', 'document_id', 'is_rare_disease']):
                continue
            
            entity = annotation['entity']
            doc_id = annotation['document_id']
            is_rare = annotation['is_rare_disease']
            context = annotation.get('context', '')
            orpha_code = annotation.get('orpha_code', '') or annotation.get('orpha_id', '')
            
            if not is_valid_annotation(entity, context):
                continue
            
            # Check if document exists in our dataset
            if doc_id not in modified_dataset:
                continue
            
            # Normalize the entity
            normalized_entity = normalize_entity(entity)
            
            # Group by (doc_id, normalized_entity)
            key = (doc_id, normalized_entity)
            entity_corrections[key].append({
                'source': source_name,
                'entity': entity,
                'is_rare': is_rare,
                'context': context,
                'orpha_code': orpha_code
            })
            collected_count += 1
        
        print(f"  Collected {collected_count} valid {source_name} corrections")
    
    # Collect from both sources
    collect_corrections(human_corrections, "Human")
    collect_corrections(hybrid_corrections, "Hybrid")
    
    print(f"\nTotal unique (document, entity) pairs to process: {len(entity_corrections)}")
    
    # Step 2: Apply OR logic to determine final classification for each entity
    print(f"\n--- Applying OR logic to corrections ---")
    
    additions_count = 0
    removals_count = 0
    updates_count = 0
    documents_modified = set()
    
    for (doc_id, normalized_entity), corrections_list in entity_corrections.items():
        # Apply OR logic: if ANY correction says is_rare=True, then final result is True
        final_is_rare = any(correction['is_rare'] for correction in corrections_list)
        
        # Get the best representative annotation (prefer one with orpha_code)
        best_correction = corrections_list[0]  # Default to first
        for correction in corrections_list:
            if correction['orpha_code']:  # Prefer one with ORPHA code
                best_correction = correction
                break
        
        # Check current state in the dataset
        if 'annotations' not in modified_dataset[doc_id]:
            modified_dataset[doc_id]['annotations'] = []
        
        # Find existing annotation with same normalized entity
        existing_annotation_idx = None
        for idx, existing_ann in enumerate(modified_dataset[doc_id]['annotations']):
            if 'mention' in existing_ann:
                existing_normalized = normalize_entity(existing_ann['mention'])
                if existing_normalized == normalized_entity:
                    existing_annotation_idx = idx
                    break
        
        # Show OR logic in action
        rare_votes = [c['is_rare'] for c in corrections_list]
        sources = [c['source'] for c in corrections_list]
        print(f"  Doc {doc_id}, '{best_correction['entity']}': {rare_votes} from {sources} -> OR = {final_is_rare}")
        
        if final_is_rare:
            # ADD or UPDATE annotation (based on OR logic result)
            new_annotation = {
                'mention': best_correction['entity'],
                'context': best_correction['context'],
                'is_rare_disease': True
            }
            
            # Add ORPHA code if available
            if best_correction['orpha_code']:
                new_annotation['orpha_code'] = best_correction['orpha_code']
                new_annotation['ordo_with_desc'] = best_correction['orpha_code']  # Maintain compatibility
            
            if existing_annotation_idx is not None:
                # Update existing annotation
                modified_dataset[doc_id]['annotations'][existing_annotation_idx].update(new_annotation)
                updates_count += 1
                print(f"    -> Updated existing annotation")
            else:
                # Add new annotation
                modified_dataset[doc_id]['annotations'].append(new_annotation)
                additions_count += 1
                print(f"    -> Added new annotation")
            
            documents_modified.add(doc_id)
            
        else:
            # REMOVE annotation if it exists (all corrections said False)
            if existing_annotation_idx is not None:
                removed_annotation = modified_dataset[doc_id]['annotations'].pop(existing_annotation_idx)
                removals_count += 1
                documents_modified.add(doc_id)
                print(f"    -> Removed annotation (all corrections said False)")
            else:
                print(f"    -> No existing annotation to remove")
    
    # Final statistics
    print(f"\n=== SET-BASED CORRECTION SUMMARY (OR LOGIC) ===")
    print(f"Entity-document pairs processed: {len(entity_corrections)}")
    print(f"Annotations added: {additions_count}")
    print(f"Annotations updated: {updates_count}")
    print(f"Annotations removed: {removals_count}")
    print(f"Documents modified: {len(documents_modified)}")
    
    # Compute before/after statistics
    def count_annotations(dataset):
        total_docs = len(dataset)
        total_annotations = 0
        docs_with_annotations = 0
        
        for doc_id, doc_data in dataset.items():
            if 'annotations' in doc_data and doc_data['annotations']:
                docs_with_annotations += 1
                total_annotations += len(doc_data['annotations'])
        
        return total_docs, total_annotations, docs_with_annotations
    
    initial_docs, initial_annotations, initial_annotated_docs = count_annotations(initial_dataset)
    final_docs, final_annotations, final_annotated_docs = count_annotations(modified_dataset)
    
    print(f"\nBefore corrections:")
    print(f"  Documents: {initial_docs}")
    print(f"  Total annotations: {initial_annotations}")
    print(f"  Documents with annotations: {initial_annotated_docs}")
    
    print(f"\nAfter corrections:")
    print(f"  Documents: {final_docs}")
    print(f"  Total annotations: {final_annotations}")
    print(f"  Documents with annotations: {final_annotated_docs}")
    print(f"  Net change in annotations: {final_annotations - initial_annotations}")
    
    return modified_dataset


def create_corrected_initial_datasets(
    initial_dataset_path: str,
    human_corrections_path: str = None,
    hybrid_corrections_path: str = None,
    human_only_output_path: str = None,
    rdma_human_output_path: str = None
):
    """
    Complete workflow to create TWO corrected versions of the initial dataset:
    1. Human corrections only
    2. RDMA & Human corrections combined
    
    Args:
        initial_dataset_path: Path to the initial filtered dataset
        human_corrections_path: Path to human corrections file
        hybrid_corrections_path: Path to RDMA+human hybrid corrections file  
        human_only_output_path: Path to save human-only corrected dataset
        rdma_human_output_path: Path to save RDMA+human corrected dataset
    
    Returns:
        Tuple of (human_only_dataset, rdma_human_dataset)
    """
    import json
    
    # Load files (reusing your existing read_json_file function)
    print("=== LOADING DATASETS ===")
    initial_dataset = read_json_file(initial_dataset_path)
    human_corrections = read_json_file(human_corrections_path) if human_corrections_path else None
    hybrid_corrections = read_json_file(hybrid_corrections_path) if hybrid_corrections_path else None
    
    if not initial_dataset:
        raise ValueError(f"Could not load initial dataset from {initial_dataset_path}")
    
    print(f"Loaded initial dataset: {len(initial_dataset)} documents")
    if human_corrections:
        correction_count = len(human_corrections.get('corrected_annotations', []))
        print(f"Loaded human corrections: {correction_count} annotations")
    if hybrid_corrections:
        if 'corrected_annotations' in hybrid_corrections:
            correction_count = len(hybrid_corrections['corrected_annotations'])
        elif 'results' in hybrid_corrections:
            correction_count = sum(len(hybrid_corrections['results'].get(cat, [])) 
                                 for cat in ['true_positives', 'false_negatives', 'false_positives'])
        else:
            correction_count = 0
        print(f"Loaded RDMA+human corrections: {correction_count} annotations")
    
    # Create human-only corrected dataset
    print("\n" + "="*80)
    print("CREATING HUMAN-ONLY CORRECTED DATASET")
    print("="*80)
    
    human_only_dataset = apply_corrections_to_initial_dataset(
        initial_dataset,
        human_corrections=human_corrections,
        hybrid_corrections=None  # No hybrid corrections for human-only
    )
    
    # Save human-only dataset
    if human_only_output_path:
        print(f"\nSaving human-only corrected dataset to {human_only_output_path}")
        with open(human_only_output_path, 'w', encoding='utf-8') as f:
            json.dump(human_only_dataset, f, indent=2, ensure_ascii=False)
        print("Human-only dataset saved successfully")
    
    # Create RDMA+human corrected dataset  
    print("\n" + "="*80)
    print("CREATING RDMA & HUMAN CORRECTED DATASET")
    print("="*80)
    
    rdma_human_dataset = apply_corrections_to_initial_dataset(
        initial_dataset,
        human_corrections=human_corrections,
        hybrid_corrections=hybrid_corrections  # Include both human and RDMA corrections
    )
    
    # Save RDMA+human dataset
    if rdma_human_output_path:
        print(f"\nSaving RDMA+human corrected dataset to {rdma_human_output_path}")
        with open(rdma_human_output_path, 'w', encoding='utf-8') as f:
            json.dump(rdma_human_dataset, f, indent=2, ensure_ascii=False)
        print("RDMA+human dataset saved successfully")
    
    return human_only_dataset, rdma_human_dataset


def create_corrected_initial_dataset(
    initial_dataset_path: str,
    human_corrections_path: str = None,
    hybrid_corrections_path: str = None,
    output_path: str = None
):
    """
    DEPRECATED: Use create_corrected_initial_datasets() instead for separate outputs.
    
    Complete workflow to create a corrected version of the initial dataset.
    This function is kept for backward compatibility.
    """
    print("Note: This function is deprecated. Consider using create_corrected_initial_datasets() for separate outputs.")
    
    # Load files (reusing your existing read_json_file function)
    print("=== LOADING DATASETS ===")
    initial_dataset = read_json_file(initial_dataset_path)
    human_corrections = read_json_file(human_corrections_path) if human_corrections_path else None
    hybrid_corrections = read_json_file(hybrid_corrections_path) if hybrid_corrections_path else None
    
    if not initial_dataset:
        raise ValueError(f"Could not load initial dataset from {initial_dataset_path}")
    
    print(f"Loaded initial dataset: {len(initial_dataset)} documents")
    if human_corrections:
        correction_count = len(human_corrections.get('corrected_annotations', []))
        print(f"Loaded human corrections: {correction_count} annotations")
    if hybrid_corrections:
        if 'corrected_annotations' in hybrid_corrections:
            correction_count = len(hybrid_corrections['corrected_annotations'])
        elif 'results' in hybrid_corrections:
            correction_count = sum(len(hybrid_corrections['results'].get(cat, [])) 
                                 for cat in ['true_positives', 'false_negatives', 'false_positives'])
        else:
            correction_count = 0
        print(f"Loaded hybrid corrections: {correction_count} annotations")
    
    # Apply corrections
    corrected_dataset = apply_corrections_to_initial_dataset(
        initial_dataset,
        human_corrections,
        hybrid_corrections
    )
    
    # Save if output path provided
    if output_path:
        import json
        print(f"\nSaving corrected dataset to {output_path}")
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(corrected_dataset, f, indent=2, ensure_ascii=False)
        print("Dataset saved successfully")
    
    return corrected_dataset


# Updated example usage function
def example_usage():
    """
    Example of how to use the correction functions to create both datasets.
    """
    print("=== EXAMPLE USAGE FOR CREATING BOTH DATASETS ===")
    
    # Your file paths (adjust as needed)
    initial_path = "data/dataset/filtered_rd_annos_enriched.json"
    human_path = "data/dataset/rare_disease_corrections_john.json"
    hybrid_path = "data/dataset/rare_disease_annotations_rdma_john_comprehensive.json"
    
    # Output paths for both versions
    human_only_output = "data/dataset/corrected_initial_dataset_human_only.json"
    rdma_human_output = "data/dataset/corrected_initial_dataset_rdma_human.json"
    
    # Create both corrected datasets
    human_only_dataset, rdma_human_dataset = create_corrected_initial_datasets(
        initial_dataset_path=initial_path,
        human_corrections_path=human_path,
        hybrid_corrections_path=hybrid_path,
        human_only_output_path=human_only_output,
        rdma_human_output_path=rdma_human_output
    )
    
    print(f"\n=== RESULTS ===")
    print(f"Human-only corrected dataset: {len(human_only_dataset)} documents")
    print(f"RDMA+human corrected dataset: {len(rdma_human_dataset)} documents")
    print(f"Files saved:")
    print(f"  - {human_only_output}")
    print(f"  - {rdma_human_output}")
    
    return human_only_dataset, rdma_human_dataset


# Integration with your existing statistics function for both datasets
def compute_both_corrected_datasets_statistics(human_only_dataset, rdma_human_dataset):
    """
    Compute statistics for both corrected datasets using your existing function.
    """
    print("=== COMPUTING STATISTICS FOR BOTH CORRECTED DATASETS ===")
    
    # Compute stats for human-only dataset
    human_only_stats = compute_stats_from_initial_format(
        human_only_dataset, 
        "Human-Only Corrected Dataset"
    )
    
    # Compute stats for RDMA+human dataset
    rdma_human_stats = compute_stats_from_initial_format(
        rdma_human_dataset, 
        "RDMA+Human Corrected Dataset"
    )
    
    # Compare the two
    print(f"\n=== COMPARISON BETWEEN CORRECTED DATASETS ===")
    if human_only_stats and rdma_human_stats:
        print(f"Human-only total rare diseases: {human_only_stats['total_unique_rare_diseases']}")
        print(f"RDMA+human total rare diseases: {rdma_human_stats['total_unique_rare_diseases']}")
        
        difference = rdma_human_stats['total_unique_rare_diseases'] - human_only_stats['total_unique_rare_diseases']
        print(f"Difference (RDMA+human - human-only): {difference}")
        
        if difference > 0:
            print(f"RDMA corrections added {difference} more rare disease annotations")
        elif difference < 0:
            print(f"RDMA corrections removed {abs(difference)} rare disease annotations")
        else:
            print("RDMA corrections made no net change to rare disease annotations")
    
    return human_only_stats, rdma_human_stats

In [5]:
# Create both datasets at once
human_only_dataset, rdma_human_dataset = create_corrected_initial_datasets(
    initial_dataset_path="data/dataset/filtered_rd_annos_enriched.json",
    human_corrections_path="data/dataset/rare_disease_corrections_john.json",
    hybrid_corrections_path="data/dataset/rare_disease_annotations_rdma_john_comprehensive.json",
    human_only_output_path="data/dataset/corrected_initial_dataset_human_only.json",
    rdma_human_output_path="data/dataset/corrected_initial_dataset_rdma_human.json"
)

=== LOADING DATASETS ===
Loaded initial dataset: 117 documents
Loaded human corrections: 333 annotations
Loaded RDMA+human corrections: 122 annotations

CREATING HUMAN-ONLY CORRECTED DATASET
=== APPLYING SET-BASED CORRECTIONS WITH OR LOGIC ===
Initial dataset documents: 117
Note: Using OR logic - if ANY correction marks entity as rare=True, it stays True

--- Collecting Human corrections ---
Found 333 Human annotations
  Collected 333 valid Human corrections
No Hybrid corrections to collect

Total unique (document, entity) pairs to process: 192

--- Applying OR logic to corrections ---
  Doc 287, 'sick sinus syndrome': [False] from ['Human'] -> OR = False
    -> Removed annotation (all corrections said False)
  Doc 869, 'legionella': [False] from ['Human'] -> OR = False
    -> Removed annotation (all corrections said False)
  Doc 1208, 'nocardiosis': [True, True] from ['Human', 'Human'] -> OR = True
    -> Updated existing annotation
  Doc 950, 'retinitis pigmentosa': [True] from ['Hum